### Pushing file from sagemaker to s3

In [1]:
import boto3
import sagemaker
from sagemaker import get_execution_role

bucket = "employee-attrition-paul"
prefix = "employee-data"
role = get_execution_role()

session = sagemaker.Session()

s3_path = session.upload_data(
    path="HR-Employee-Attrition.csv",
    bucket=bucket,
    key_prefix=prefix
)

print("Dataset uploaded:", s3_path)

### Training job

In [9]:
from sagemaker.sklearn.estimator import SKLearn
from sagemaker import get_execution_role
import sagemaker
from utils import append_to_file
session = sagemaker.Session()
role = get_execution_role()
bucket = "employee-attrition-paul"
prefix = "employee-data"
sklearn_estimator = SKLearn(
    entry_point="train.py",
    role=role,
    instance_type="ml.c5.xlarge",  # or "ml.t2.medium" for free tier
    framework_version="1.2-1",
    py_version="py3",
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=session
)
append_to_file("output.txt","framework_version 1.2-1")
# Start training
sklearn_estimator.fit(
    {
        "train": f"s3://{bucket}/{prefix}"
    }
)
image_uri=sklearn_estimator.training_image_uri()
append_to_file("output.txt","Image URI",image_uri)
model_s3_uri = sklearn_estimator.model_data
append_to_file("output.txt","model_s3_uri",model_s3_uri)
print("Model S3 URI:", model_s3_uri)

### Creating server based endpoint

In [10]:
from sagemaker.sklearn.model import SKLearnModel
model = SKLearnModel(
    model_data="s3://employee-attrition-paul/employee-data/output/sagemaker-scikit-learn-2026-07-22-05-07-53-983/output/model.tar.gz",  # <- from training
    role=role,
    entry_point="inference.py",                         # ✅ NEW inference script
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=session
)

In [11]:
predictor = model.deploy(
    instance_type="ml.c5.xlarge",
    initial_instance_count=1,
    endpoint_name="attrition-paul-endpoint2"
)

In [12]:
def predict_attrition(
    predictor,
    age,
    daily_rate,
    distance_from_home,
    employee_number,
    monthly_income,
    monthly_rate,
    total_working_years,
    years_at_company,
    years_in_current_role,
    years_with_curr_manager,
    jobrole_sales_representative,
    overtime_yes
):
    """
    Sends Employee Attrition prediction request
    to the SageMaker endpoint.
    """

    from sagemaker.serializers import JSONSerializer
    from sagemaker.deserializers import JSONDeserializer

    predictor.serializer = JSONSerializer()
    predictor.deserializer = JSONDeserializer()

    input_data = {
        "Age": [age],
        "DailyRate": [daily_rate],
        "DistanceFromHome": [distance_from_home],
        "EmployeeNumber": [employee_number],
        "MonthlyIncome": [monthly_income],
        "MonthlyRate": [monthly_rate],
        "TotalWorkingYears": [total_working_years],
        "YearsAtCompany": [years_at_company],
        "YearsInCurrentRole": [years_in_current_role],
        "YearsWithCurrManager": [years_with_curr_manager],
        "JobRole_Sales Representative": [jobrole_sales_representative],
        "OverTime_Yes": [overtime_yes]
    }

    prediction = predictor.predict(input_data)

    return prediction

In [13]:
prediction = predict_attrition(
    predictor,
    age=35,
    daily_rate=1102,
    distance_from_home=5,
    employee_number=1001,
    monthly_income=6000,
    monthly_rate=15000,
    total_working_years=10,
    years_at_company=5,
    years_in_current_role=3,
    years_with_curr_manager=2,
    jobrole_sales_representative=0,
    overtime_yes=1
)

print(prediction)

#### 0 = No Attrition (Person wil likely stay in the company)
#### 1 = Attrition (Person will likely leave the company)

In [15]:
import boto3

client = boto3.client('sagemaker', region_name='ap-south-1')
response = client.list_endpoints()
for ep in response['Endpoints']:
    print(ep['EndpointName'])


In [16]:
from sagemaker import Predictor

# Use the same credentials/session setup as before
predictor = Predictor(endpoint_name="attrition-paul-endpoint1")
predictor.delete_endpoint()
print("✅ Endpoint deleted successfully.")


In [17]:
from sagemaker import Predictor

# Use the same credentials/session setup as before
predictor = Predictor(endpoint_name="attrition-paul-endpoint2")
predictor.delete_endpoint()
print("✅ Endpoint deleted successfully.")

### Serverless Endpoint

In [18]:
from sagemaker.serverless import ServerlessInferenceConfig
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import Session, get_execution_role

session = Session()
role = get_execution_role()

model = SKLearnModel(
    model_data="s3://employee-attrition-paul/employee-data/output/sagemaker-scikit-learn-2026-07-22-05-07-53-983/output/model.tar.gz",
    role=role,
    entry_point="inference.py",
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=session
)

predictor = model.deploy(
    endpoint_name="attrition-paul-serverless-endpoint",
    serverless_inference_config=ServerlessInferenceConfig(
        memory_size_in_mb=1024,
        max_concurrency=5
    )
)

In [19]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer


def predict_attrition(
    predictor,
    age,
    daily_rate,
    distance_from_home,
    employee_number,
    monthly_income,
    monthly_rate,
    total_working_years,
    years_at_company,
    years_in_current_role,
    years_with_curr_manager,
    jobrole_sales_representative,
    overtime_yes
):

    predictor.serializer = JSONSerializer()
    predictor.deserializer = JSONDeserializer()

    input_data = {
        "Age": [age],
        "DailyRate": [daily_rate],
        "DistanceFromHome": [distance_from_home],
        "EmployeeNumber": [employee_number],
        "MonthlyIncome": [monthly_income],
        "MonthlyRate": [monthly_rate],
        "TotalWorkingYears": [total_working_years],
        "YearsAtCompany": [years_at_company],
        "YearsInCurrentRole": [years_in_current_role],
        "YearsWithCurrManager": [years_with_curr_manager],
        "JobRole_Sales Representative": [jobrole_sales_representative],
        "OverTime_Yes": [overtime_yes]
    }

    prediction = predictor.predict(input_data)

    return prediction

In [20]:
result = predict_attrition(
    predictor,
    age=35,
    daily_rate=1102,
    distance_from_home=5,
    employee_number=1001,
    monthly_income=6000,
    monthly_rate=15000,
    total_working_years=10,
    years_at_company=5,
    years_in_current_role=3,
    years_with_curr_manager=2,
    jobrole_sales_representative=0,
    overtime_yes=1
)

print("Predicted Attrition:", result)

In [21]:
from sagemaker import Predictor

# Use the same credentials/session setup as before
predictor = Predictor(endpoint_name="attrition-paul-serverless-endpoint")
predictor.delete_endpoint()
print("✅ Endpoint deleted successfully.")